<a href="https://colab.research.google.com/github/jhenningsen/Equity_Analysis/blob/main/Misc/IPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [62]:
import json
import time
import io
import requests
import pandas as pd
import yfinance as yf


### Loading and Merging 'IPO Deal Size' Data

Now, let's load the provided `IPO_Deal_Size.csv` file and merge its information (Deal Size) into our `consolidated_df`. We will assume the 'Symbol' column is present in both DataFrames for the merge operation.

In [63]:
# These are Google Drive file IDs. To get your own, right-click on the file in Google Drive, select 'Share', then 'Get link'. The ID is the part of the URL after 'id='.
IPO_2024_FileId = '1CiGIOc7Z3Td4xTCICy7H3cF0QBIeUFiu'
IPO_2024_File = f'https://drive.google.com/uc?export=download&id={IPO_2024_FileId}'
# Load the IPO_Deal_Size.txt file as a pipe-separated file, handling quoted fields
ipo_2024_df = pd.read_csv(IPO_2024_File, sep=',', quotechar='"', skipinitialspace=True)

IPO_2025_FileId = '1uSI7gY4uKACxZqipBj3SefHSouAaNfrr'
IPO_2025_File = f'https://drive.google.com/uc?export=download&id={IPO_2025_FileId}'
# Load the IPO_Deal_Size.txt file as a pipe-separated file, handling quoted fields
ipo_2025_df = pd.read_csv(IPO_2025_File, sep=',', quotechar='"', skipinitialspace=True)

IPO_2026_FileId = '1XjCKzN_pXzoawIGpFqQs2iAtdBOBbvJC'
IPO_2026_File = f'https://drive.google.com/uc?export=download&id={IPO_2026_FileId}'
# Load the IPO_Deal_Size.txt file as a pipe-separated file, handling quoted fields
ipo_2026_df = pd.read_csv(IPO_2026_File, sep=',', quotechar='"', skipinitialspace=True)

In [64]:
# Concatenate the three IPO DataFrames into a single DataFrame
consolidated_df_with_deal_size = pd.concat([ipo_2024_df, ipo_2025_df, ipo_2026_df], ignore_index=True)

print("--- Consolidated DataFrame with Deal Size (Head) ---")
display(consolidated_df_with_deal_size.head())

print("--- Consolidated DataFrame with Deal Size (Tail) ---")
display(consolidated_df_with_deal_size.tail())

print("\nMerge complete. The `consolidated_df_with_deal_size` DataFrame now includes IPO data from 2024, 2025, and 2026.")

--- Consolidated DataFrame with Deal Size (Head) ---


,IPO Date,Symbol,Company Name,Current,Return,Exchange,Deal Size,Shares Offered,IPO Price
0,"Dec 31, 2024",ONEG,OneConstruction Group Limited,$1.20,-71.00%,NASDAQ,7.00M,"1,750,000",$4.00
1,"Dec 27, 2024",BYAH,"Park Ha Biological Technology Co., Ltd.",$0.435,-,NASDAQ,-,-,-
2,"Dec 23, 2024",HIT,"Health In Tech, Inc.",$1.03,-73.50%,NASDAQ,9.20M,"2,300,000",$4.00
3,"Dec 23, 2024",TDAC,Translational Development Acquisition Corp.,$10.70,7.00%,NASDAQ,150.00M,"15,000,000",$10.00
4,"Dec 20, 2024",RANG,Range Capital Acquisition Corp.,$10.67,6.70%,NASDAQ,100.00M,"10,000,000",$10.00


--- Consolidated DataFrame with Deal Size (Tail) ---


,IPO Date,Symbol,Company Name,Current,Return,Exchange,Deal Size,Shares Offered,IPO Price
766,"Jan 8, 2026",BBCQ,Bleichroeder Acquisition Corp. II,$10.32,3.20%,NASDAQ,"25,000,000",250.00M,$10.00
767,"Jan 8, 2026",BUDA,"Buda Juice, Inc.",$10.80,43.20%,NYSEAMERICAN,"2,666,667",20.00M,$7.50
768,"Jan 7, 2026",SORN,Soren Acquisition Corp.,$9.94,-0.60%,NASDAQ,"22,000,000",220.00M,$10.00
769,"Jan 6, 2026",ARTC,Art Technology Acquisition Corp.,$9.98,-0.20%,NASDAQ,"22,000,000",220.00M,$10.00
770,"Jan 6, 2026",BIII,Black Spade Acquisition III Co,$9.94,-0.40%,NYSE,"15,000,000",150.00M,$10.00



Merge complete. The `consolidated_df_with_deal_size` DataFrame now includes IPO data from 2024, 2025, and 2026.


### Determine the First Trading Day After IPO

To ensure we calculate returns from the correct starting point, we'll identify the actual first trading day after each IPO date. The `yfinance.download()` function automatically handles non-trading days (weekends, holidays) by returning data from the first available trading day on or after the specified `start` date.

In [69]:
from tqdm.notebook import tqdm

# Ensure 'IPO Date' in consolidated_df_with_deal_size is datetime type
consolidated_df_with_deal_size['IPO Date'] = pd.to_datetime(consolidated_df_with_deal_size['IPO Date'])

# First, filter by 'Deal Size' containing 'B' as requested
deal_size_filtered_df = consolidated_df_with_deal_size[consolidated_df_with_deal_size['Deal Size'].str.contains('B', na=False, case=False)].copy()

next_trading_days = []

print(f"Calculating the first trading day after IPO for {len(deal_size_filtered_df)} stocks using the specified window...")
for index, row in tqdm(deal_size_filtered_df.iterrows(), total=len(deal_size_filtered_df)):
    symbol = row['Symbol']
    ipo_date = row['IPO Date']

    # Start fetching data from the calendar day after IPO
    effective_start_date = ipo_date + timedelta(days=1)

    try:
        # Fetch data for a 15-calendar-day window to reliably find the actual first trading day
        ticker_data = yf.download(symbol,
                                  start=effective_start_date.strftime('%Y-%m-%d'),
                                  end=(effective_start_date + timedelta(days=15)).strftime('%Y-%m-%d'), # Check for next 15 calendar days
                                  interval="1d",
                                  progress=False,
                                  auto_adjust=False) # Explicitly set auto_adjust to False

        if not ticker_data.empty and 'Adj Close' in ticker_data.columns:
            # Select the minimum date from the downloaded dataset as the first trading day
            next_trading_days.append(ticker_data.index.min().normalize()) # Get just the date part
        else:
            next_trading_days.append(pd.NaT) # No trading data found
            print(f"    ⚠️ Could not find trading data for {symbol} starting from {effective_start_date.strftime('%Y-%m-%d')} within 15 calendar days for First_Trading_Day_After_IPO.")

    except Exception as e:
        # Handle cases where yf.download itself might fail (e.g., invalid symbol)
        next_trading_days.append(pd.NaT)
        print(f"    ❌ Error determining next trading day for {symbol} after {ipo_date.strftime('%Y-%m-%d')}: {e}")

deal_size_filtered_df['First_Trading_Day_After_IPO'] = next_trading_days

print("\n--- Deal Size Filtered DataFrame with First Trading Day After IPO (Head) ---")
display(deal_size_filtered_df[['Symbol', 'IPO Date', 'Deal Size', 'First_Trading_Day_After_IPO']].head())

print("\n--- Deal Size Filtered DataFrame with First Trading Day After IPO (Tail) ---")
display(deal_size_filtered_df[['Symbol', 'IPO Date', 'Deal Size', 'First_Trading_Day_After_IPO']].tail())

Calculating the first trading day after IPO for 13 stocks using the specified window...


  0%|          | 0/13 [00:00<?, ?it/s]


--- Deal Size Filtered DataFrame with First Trading Day After IPO (Head) ---


,Symbol,IPO Date,Deal Size,First_Trading_Day_After_IPO
69,SARO,2024-10-02,1.44B,2024-10-03
115,LINE,2024-07-25,4.44B,2024-07-26
160,VIK,2024-05-01,1.27B,2024-05-02
209,AS,2024-02-01,1.37B,2024-02-02
234,MDLN,2025-12-17,6.26B,2025-12-18



--- Deal Size Filtered DataFrame with First Trading Day After IPO (Tail) ---


,Symbol,IPO Date,Deal Size,First_Trading_Day_After_IPO
380,NIQ,2025-07-23,1.05B,2025-07-24
431,CRCL,2025-06-05,1.05B,2025-06-06
499,CRWV,2025-03-28,1.50B,2025-03-31
532,SAIL,2025-02-13,1.38B,2025-02-14
553,VG,2025-01-24,1.75B,2025-01-27


Let's specifically check for 'MDLN' data on Yahoo Finance starting from 2025-12-18.

Let's inspect the `Deal Size` column in `consolidated_df_with_deal_size` to understand its format and why the filter for 'B' might not be capturing all expected entries.

In [66]:
print("Value counts for 'Deal Size' column in `consolidated_df_with_deal_size`:")
display(consolidated_df_with_deal_size['Deal Size'].value_counts(dropna=False).head(20))

print("\nSample of rows where 'Deal Size' contains 'B' (case-insensitive):")
display(consolidated_df_with_deal_size[consolidated_df_with_deal_size['Deal Size'].str.contains('B', na=False, case=False)].head(20))

Value counts for 'Deal Size' column in `consolidated_df_with_deal_size`:


,count
Deal Size,
200.00M,47
-,35
"20,000,000",30
150.00M,25
5.00M,21
8.00M,20
"10,000,000",19
6.00M,16
60.00M,15



Sample of rows where 'Deal Size' contains 'B' (case-insensitive):


,IPO Date,Symbol,Company Name,Current,Return,Exchange,Deal Size,Shares Offered,IPO Price
69,2024-10-02,SARO,"StandardAero, Inc.",$27.77,15.58%,NYSE,1.44B,"60,000,000",$24.00
115,2024-07-25,LINE,"Lineage, Inc.",$44.00,-43.78%,NASDAQ,4.44B,"56,882,051",$78.00
160,2024-05-01,VIK,Viking Holdings Ltd,$98.65,311.25%,NYSE,1.27B,"53,000,000",$24.00
209,2024-02-01,AS,"Amer Sports, Inc.",$34.18,164.39%,NYSE,1.37B,"105,000,000",$13.00
234,2025-12-17,MDLN,Medline Inc.,$39.15,34.76%,NASDAQ,6.26B,"216,034,482",$29.00
271,2025-11-04,BETA,"BETA Technologies, Inc.",$18.19,-47.06%,NYSE,1.01B,"29,852,941",$34.00
335,2025-09-10,KLAR,Klarna Group plc,$19.39,-51.85%,NYSE,1.37B,"34,311,274",$40.00
356,2025-08-13,BLSH,Bullish,$25.08,-34.46%,NYSE,1.11B,"30,000,000",$37.00
380,2025-07-23,NIQ,NIQ Global Intelligence plc,$11.11,-47.29%,NYSE,1.05B,"50,000,000",$21.00
431,2025-06-05,CRCL,"Circle Internet Group, Inc.",$63.05,101.00%,NYSE,1.05B,"34,000,000",$31.00


### Analyze and Calculate Returns for Large Deal Sizes

First, I will filter the `consolidated_df_with_deal_size` DataFrame to include only those IPOs where the 'Deal Size' contains 'B' (indicating billions). Then, for each of these filtered IPOs, I will use the `yfinance` library to fetch the stock's historical data for the IPO date. Finally, I will calculate the return based on the 'Open' and 'Close' prices for that specific date.

In [74]:
import pandas as pd
import yfinance as yf
from datetime import timedelta, datetime # Import datetime

# The base DataFrame is now deal_size_filtered_df which already has 'Deal Size' filtered
# and 'First_Trading_Day_After_IPO' calculated.

# Ensure 'IPO Date' column is converted to datetime if not already (already done, but good practice)
deal_size_filtered_df['IPO Date'] = pd.to_datetime(deal_size_filtered_df['IPO Date'])

# Filter out IPOs that are in the future based on the current 'today' date
today = datetime.now().date() # Get only the date part of today
print(f"Current 'today' date used for filtering: {today}")

# Apply the remaining filters: IPOs in the past and those with a valid First_Trading_Day_After_IPO
filtered_df = deal_size_filtered_df[
    (deal_size_filtered_df['IPO Date'].dt.date <= today) &
    (deal_size_filtered_df['First_Trading_Day_After_IPO'].notna())
].copy()

# Prepare empty lists for 1, 3, 5, 10, 20, 30 day returns
returns_1d = []
returns_3d = []
returns_5d = []
returns_10d = []
returns_20d = []
returns_30d = []

print(f"Found {len(filtered_df)} *past* IPOs with 'B' in 'Deal Size' and valid 'First_Trading_Day_After_IPO'. Attempting to fetch historical stock data for post-IPO returns...")

for index, row in filtered_df.iterrows():
    symbol = row['Symbol']
    ipo_date = row['IPO Date'] # Original IPO Date
    first_trading_day_after_ipo = row['First_Trading_Day_After_IPO'] # Actual first trading day from previous step

    try:
        # The effective_start_date is now directly from the 'First_Trading_Day_After_IPO' column
        effective_start_date = first_trading_day_after_ipo

        # Fetch data for a period long enough to cover 30 trading days after IPO
        # 60 calendar days should be sufficient to capture 30 trading days from effective_start_date
        end_fetch_date = effective_start_date + timedelta(days=60)

        # Download daily historical data starting from the actual first trading day after IPO
        ticker_data = yf.download(symbol,
                                  start=effective_start_date.strftime('%Y-%m-%d'),
                                  end=end_fetch_date.strftime('%Y-%m-%d'),
                                  interval="1d",
                                  progress=False,
                                  auto_adjust=False) # Explicitly set auto_adjust to False here too

        # Check if ticker_data is not empty and contains 'Adj Close'
        # This check is now mostly redundant for the first entry due to previous filtering,
        # but still useful for ensuring enough data for subsequent days_offset.
        if not ticker_data.empty and 'Adj Close' in ticker_data.columns:
            first_trading_day_close = ticker_data['Adj Close'].iloc[0].item() # Use .item() to ensure scalar
            actual_data_start_date = ticker_data.index[0] # Should be close to effective_start_date

            # Calculate returns for specific trading days
            trading_days_to_check = [
                ('Return_1_Trading_Day', 1),
                ('Return_3_Trading_Days', 3),
                ('Return_5_Trading_Days', 5),
                ('Return_10_Trading_Days', 10),
                ('Return_20_Trading_Days', 20),
                ('Return_30_Trading_Days', 30)
            ]

            current_returns = {label: None for label, _ in trading_days_to_check}

            for label, days_offset in trading_days_to_check:
                # Ensure there are enough trading days to get the offset price
                if len(ticker_data) > days_offset:
                    price_at_offset = ticker_data['Adj Close'].iloc[days_offset].item() # Use .item() to ensure scalar
                    if first_trading_day_close != 0: # Avoid division by zero
                        ret = (price_at_offset - first_trading_day_close) / first_trading_day_close
                        current_returns[label] = ret
                # If not enough trading days, it remains None as initialized

            returns_1d.append(current_returns['Return_1_Trading_Day'])
            returns_3d.append(current_returns['Return_3_Trading_Days'])
            returns_5d.append(current_returns['Return_5_Trading_Days'])
            returns_10d.append(current_returns['Return_10_Trading_Days'])
            returns_20d.append(current_returns['Return_20_Trading_Days'])
            returns_30d.append(current_returns['Return_30_Trading_Days'])

            print(f"    ✅ Fetched {symbol} (IPO Date: {ipo_date.strftime('%Y-%m-%d')}). Actual first trading day: {actual_data_start_date.strftime('%Y-%m-%d')}.")

        else:
            # This else block should ideally not be hit often if First_Trading_Day_After_IPO is notna()
            print(f"    ❌ No valid historical stock data (or 'Adj Close' column missing) found for {symbol} starting from {effective_start_date.strftime('%Y-%m-%d')} from Yahoo Finance, even after pre-filtering.")
            returns_1d.append(None)
            returns_3d.append(None)
            returns_5d.append(None)
            returns_10d.append(None)
            returns_20d.append(None)
            returns_30d.append(None)

    except Exception as e:
        print(f"    ❌ Error fetching data for {symbol} (IPO Date: {ipo_date.strftime('%Y-%m-%d')}): {e}")
        returns_1d.append(None)
        returns_3d.append(None)
        returns_5d.append(None)
        returns_10d.append(None)
        returns_20d.append(None)
        returns_30d.append(None)

# Add the calculated returns to the filtered DataFrame
filtered_df['Return_1_Trading_Day'] = returns_1d
filtered_df['Return_3_Trading_Days'] = returns_3d
filtered_df['Return_5_Trading_Days'] = returns_5d
filtered_df['Return_10_Trading_Days'] = returns_10d
filtered_df['Return_20_Trading_Days'] = returns_20d
filtered_df['Return_30_Trading_Days'] = returns_30d

print("--- Filtered DataFrame with Calculated Returns (Head) ---")
display(filtered_df[['Symbol', 'Company Name', 'IPO Date', 'First_Trading_Day_After_IPO', 'Deal Size',
                     'Return_1_Trading_Day', 'Return_3_Trading_Days', 'Return_5_Trading_Days',
                     'Return_10_Trading_Days', 'Return_20_Trading_Days', 'Return_30_Trading_Days']].head())

print("\n--- Filtered DataFrame with Calculated Returns (Tail) ---")
display(filtered_df[['Symbol', 'Company Name', 'IPO Date', 'First_Trading_Day_After_IPO', 'Deal Size',
                     'Return_1_Trading_Day', 'Return_3_Trading_Days', 'Return_5_Trading_Days',
                     'Return_10_Trading_Days', 'Return_20_Trading_Days', 'Return_30_Trading_Days']].tail())

Current 'today' date used for filtering: 2026-07-14
Found 13 *past* IPOs with 'B' in 'Deal Size' and valid 'First_Trading_Day_After_IPO'. Attempting to fetch historical stock data for post-IPO returns...
    ✅ Fetched SARO (IPO Date: 2024-10-02). Actual first trading day: 2024-10-03.
    ✅ Fetched LINE (IPO Date: 2024-07-25). Actual first trading day: 2024-07-26.
    ✅ Fetched VIK (IPO Date: 2024-05-01). Actual first trading day: 2024-05-02.
    ✅ Fetched AS (IPO Date: 2024-02-01). Actual first trading day: 2024-02-02.
    ✅ Fetched MDLN (IPO Date: 2025-12-17). Actual first trading day: 2025-12-18.
    ✅ Fetched BETA (IPO Date: 2025-11-04). Actual first trading day: 2025-11-05.
    ✅ Fetched KLAR (IPO Date: 2025-09-10). Actual first trading day: 2025-09-11.
    ✅ Fetched BLSH (IPO Date: 2025-08-13). Actual first trading day: 2025-08-14.
    ✅ Fetched NIQ (IPO Date: 2025-07-23). Actual first trading day: 2025-07-24.
    ✅ Fetched CRCL (IPO Date: 2025-06-05). Actual first trading day: 20

,Symbol,Company Name,IPO Date,First_Trading_Day_After_IPO,Deal Size,Return_1_Trading_Day,Return_3_Trading_Days,Return_5_Trading_Days,Return_10_Trading_Days,Return_20_Trading_Days,Return_30_Trading_Days
69,SARO,"StandardAero, Inc.",2024-10-02,2024-10-03,1.44B,-0.014255,-0.002427,-0.041553,-0.008493,-0.124962,-0.178041
115,LINE,"Lineage, Inc.",2024-07-25,2024-07-26,4.44B,0.037524,0.060328,0.037524,0.061414,0.035473,-0.000844
160,VIK,Viking Holdings Ltd,2024-05-01,2024-05-02,1.27B,0.074472,0.061504,0.055947,0.040015,0.163764,0.143386
209,AS,"Amer Sports, Inc.",2024-02-01,2024-02-02,1.37B,-0.023411,0.006689,-0.000669,0.068227,0.159866,0.007358
234,MDLN,Medline Inc.,2025-12-17,2025-12-18,6.26B,0.054430,0.104557,0.117215,-0.012658,0.117722,0.132152



--- Filtered DataFrame with Calculated Returns (Tail) ---


,Symbol,Company Name,IPO Date,First_Trading_Day_After_IPO,Deal Size,Return_1_Trading_Day,Return_3_Trading_Days,Return_5_Trading_Days,Return_10_Trading_Days,Return_20_Trading_Days,Return_30_Trading_Days
380,NIQ,NIQ Global Intelligence plc,2025-07-23,2025-07-24,1.05B,-0.004559,-0.046606,-0.065856,-0.124620,-0.105876,-0.137285
431,CRCL,"Circle Internet Group, Inc.",2025-06-05,2025-06-06,1.05B,0.070102,0.088208,0.240111,1.446147,0.901671,0.841319
499,CRWV,"CoreWeave, Inc.",2025-03-28,2025-03-31,1.50B,0.417745,0.450917,0.344390,0.176106,0.154800,0.706041
532,SAIL,"SailPoint, Inc.",2025-02-13,2025-02-14,1.38B,0.046843,-0.000815,-0.048880,-0.039919,-0.192668,-0.236253
553,VG,"Venture Global, Inc.",2025-01-24,2025-01-27,1.75B,0.003512,0.022579,0.075765,-0.136478,-0.292022,-0.513906


In [77]:
print("\n--- Summary Statistics for IPO Returns ---")

return_columns = [
    'Return_1_Trading_Day',
    'Return_3_Trading_Days',
    'Return_5_Trading_Days',
    'Return_10_Trading_Days',
    'Return_20_Trading_Days',
    'Return_30_Trading_Days'
]

summary_stats = pd.DataFrame(columns=['Average Return', 'Median Return', 'Win Rate'])

for col in return_columns:
    avg_return = filtered_df[col].mean()
    median_return = filtered_df[col].median()

    # Calculate Win Rate
    positive_returns_count = (filtered_df[col] > 0).sum()
    total_ipos = len(filtered_df)
    if total_ipos > 0:
        win_rate = positive_returns_count / total_ipos
    else:
        win_rate = 0.0 # Handle case where there are no IPOs

    summary_stats.loc[col] = [avg_return, median_return, win_rate]

display(summary_stats.style.format({
    'Average Return': "{:.2%}",
    'Median Return': "{:.2%}",
    'Win Rate': "{:.2%}"
}))

print("\nThese statistics provide a quick overview of the central tendency and success rate of returns for the selected IPOs over different trading periods.")


--- Summary Statistics for IPO Returns ---


,Average Return,Median Return,Win Rate
Return_1_Trading_Day,4.33%,0.42%,61.54%
Return_3_Trading_Days,4.08%,2.26%,61.54%
Return_5_Trading_Days,5.23%,3.75%,53.85%
Return_10_Trading_Days,7.49%,-1.27%,38.46%
Return_20_Trading_Days,2.23%,-3.37%,46.15%
Return_30_Trading_Days,2.20%,-13.43%,38.46%



These statistics provide a quick overview of the central tendency and success rate of returns for the selected IPOs over different trading periods.
